In [ ]:
import subprocess
import os
import requests

file_paths = [
]

base_url = 'https://raw.githubusercontent.com/'

llvm_ir_outputs = {}

SRC_ROOT = "/content/dataset"
DST_ROOT = "/content/dataset_ll"

os.makedirs(SRC_ROOT, exist_ok=True)
os.makedirs(DST_ROOT, exist_ok=True)

os.chdir(SRC_ROOT)

for file_path in file_paths:
    print(f"\nProcessing {file_path}...")

    url = base_url + file_path
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Error: Could not fetch {file_path} (Status code: {response.status_code})")
        continue

    c_code = response.text

    base_name = os.path.splitext(file_path.replace('/', '_'))[0]
    temp_c_file = f"{base_name}.c"
    with open(temp_c_file, "w") as f:
        f.write(c_code)

    ll_file = f"{DST_ROOT}/{base_name}.ll"

    try:
        subprocess.run(["clang", "-S", "-emit-llvm", "-O0", temp_c_file, "-o", ll_file], check=True) # No optimization applied to the code
        print(f"LLVM IR generated for {file_path}.")

        # Read the generated LLVM IR
        with open(ll_file, "r") as f:
            llvm_ir = f.read()
        llvm_ir_outputs[file_path] = llvm_ir

    except subprocess.CalledProcessError as e:
        print(f"Error generating LLVM IR for {file_path}: {e}")
    except FileNotFoundError:
        print(f"Error: {ll_file} not found.")

# for file_path, llvm_ir in llvm_ir_outputs.items():
    # print(f"\nLLVM IR for {file_path}:\n{'='*80}\n{llvm_ir}\n{'='*80}")


Processing conversions/celsius_to_fahrenheit.c...
LLVM IR generated for conversions/celsius_to_fahrenheit.c.

Processing conversions/binary_to_decimal.c...
LLVM IR generated for conversions/binary_to_decimal.c.

Processing conversions/binary_to_hexadecimal.c...
LLVM IR generated for conversions/binary_to_hexadecimal.c.

Processing conversions/binary_to_octal.c...
LLVM IR generated for conversions/binary_to_octal.c.

Processing conversions/decimal_to_binary.c...
LLVM IR generated for conversions/decimal_to_binary.c.

Processing conversions/decimal_to_binary_recursion.c...
LLVM IR generated for conversions/decimal_to_binary_recursion.c.

Processing conversions/decimal_to_hexa.c...
LLVM IR generated for conversions/decimal_to_hexa.c.

Processing conversions/decimal_to_octal.c...
LLVM IR generated for conversions/decimal_to_octal.c.

Processing conversions/decimal_to_octal_recursion.c...
LLVM IR generated for conversions/decimal_to_octal_recursion.c.

Processing conversions/int_to_string.

In [2]:
!pip install transformers accelerate bitsandbytes
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 14.5 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

login("TOKEN")

In [7]:
import os
from glob import glob
from pathlib import Path
from textwrap import indent
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAMES = ["facebook/llm-compiler-7b-ftd"]

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

class LLM_Compiler:
    def __init__(self, model_name: str = "facebook/llm-compiler-7b-ftd", device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        if model_name not in MODEL_NAMES:
            raise ValueError(f"model_name must be one of {MODEL_NAMES}")
        self.model_name = model_name
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=quant_config,
            device_map="auto"
        )
        self.model.eval()

    def infer(self, prompt: str, max_new_tokens: int = 50) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        text: str = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return text[len(prompt):]

    def get_tokenizer(self):
        return self.tokenizer

    def optimize_for_code_size(self, ir: str, max_new_tokens: int = 50) -> str:
        prompt = f"""[INST] Tell me how to optimize this LLVM-IR for object file size:
<code>{ir}</code> [/INST]"""
        return self.infer(prompt, max_new_tokens=max_new_tokens)

if __name__ == "__main__":
    # Define paths
    DST_ROOT = "/content/dataset_ll"

    DST_ROOT = "/content/dataset_ll"
    OUTPUT_DIR = "/content/dataset_ll_outputs"
    max_new_tokens = 3500
    max_tokens1 = 1850  # Max tokens for IR + prompt
    max_tokens2 = 3500
    # max_tokens = 1850

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    llm_compiler = LLM_Compiler()
    tokenizer = llm_compiler.get_tokenizer()

    # Load .ll files
    ll_files = glob(f"{DST_ROOT}/*.ll")[:100]
    if not ll_files:
        print(f"No .ll files found in {DST_ROOT}. Run the data loading script first.")
        exit()

    print(f"\nProcessing up to {len(ll_files)} LLVM-IR samples from {DST_ROOT}\n")

    for ll_file in ll_files:
        if 'sample' in Path(ll_file).stem:
          continue
        with open(ll_file, 'r') as f:
            ir = f.read()
        if 'define' not in ir:
            print(f"Skipping {ll_file}: Invalid or empty LLVM-IR")
            continue
        if os.path.getsize(ll_file) > 10000:  # Skip files >10KB
            print(f"Skipping {ll_file}: File too large")
            continue

        prompt = f"""[INST] Tell me how to optimize this LLVM-IR for object file size:
<code>{ir}</code> [/INST]"""
        token_count = len(tokenizer.encode(prompt))
        if token_count > max_tokens2 or token_count < max_tokens1:
        # if token_count > max_tokens:
            # print(f"Skipping {ll_file}: {token_count} tokens exceed limit of {max_tokens}")
            print(f"Skipping {ll_file}: {token_count} tokens exceed limit")
            continue

        file_name = Path(ll_file).stem
        print(f"\nProcessing {file_name}.ll ({token_count} tokens):")

        detailed = llm_compiler.optimize_for_code_size(ir, max_new_tokens)
        detailed_output_file = f"{OUTPUT_DIR}/{file_name}_opt_run1.ll"
        with open(detailed_output_file, 'w') as f:
            f.write(detailed)
        print(f"Saved detailed optimizations to {detailed_output_file}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Processing up to 29 LLVM-IR samples from /content/dataset_ll


Processing math_strong_number.ll (2643 tokens):


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saved detailed optimizations to /content/dataset_ll_outputs/math_strong_number_opt_run1.ll

Processing conversions_decimal_to_binary.ll (2554 tokens):


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saved detailed optimizations to /content/dataset_ll_outputs/conversions_decimal_to_binary_opt_run1.ll

Processing math_lcm.ll (1877 tokens):


KeyboardInterrupt: 